In [1]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
df_results = pd.read_csv("quantization_tiny_aya_results.csv")
NUM_LAYERS = 36


### Step 1: Environment Configuration & ROCm Execution Sanity Check

We initialize the execution environment on AMD Strix Halo using ROCm. We configure PyTorch to use `HSA_OVERRIDE_GFX_VERSION=11.0.0` to target the RDNA 3.5 iGPU architecture cleanly, disable tokenizers parallelism to eliminate fork warnings, and assert CUDA/ROCm device availability.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")


PyTorch Version: 2.5.1+rocm6.2
Device: cuda
Device Name: AMD Radeon Graphics
Allocated Memory: 0.00 MB


amdgpu.ids: No such file or directory


### Step 2: Parallel Multi-Hop Queries, Prefill Framing & Tokenizer Diagnostics

To enable rigorous logit lens evaluation on Tiny Aya without definite-article ambiguity, we format our 2-hop reasoning queries using Aya's native chat template with an assistant response prefill ending in a trailing space:
- **English:** `... <|START_RESPONSE|>The capital is `
- **MSA:** `... <|START_RESPONSE|>العاصمة هي `
- **Masri:** `... <|START_RESPONSE|>عاصمة البلد هي `

**Tokenizer Diagnostic Insight:**
Tiny Aya uses a 262,144-vocabulary `Cohere2` BPE tokenizer. Because BPE treats leading whitespace as part of the token:
- Leading space + Cairo (`' القاهرة'`) maps to a **single unified token ID `97936`**.
- Leading space + Cairo in English (`' Cairo'`) maps to a **single unified token ID `147633`**.
- Competing cities decompose into distinct initial tokens (`' الإ'` for Alexandria, `' الس'` for Suez, `' الجيزة'` for Giza).

Below, we construct the prompts and inspect the tokenization diagnostics.

In [2]:
TOKENIZER_NAME = "CohereLabs/tiny-aya-earth"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

RAW_QUERIES = {
    "EN": ("What is the capital of the country where the Suez Canal is located?", "The capital is "),
    "MSA": ("ما هي عاصمة الدولة التي تقع فيها قناة السويس؟", "العاصمة هي "),
    "Masri": ("ايه هي عاصمة البلد اللي فيها قناة السويس؟", "عاصمة البلد هي "),
}

PROMPTS = {}
print("=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===")
for lang, (q, prefill) in RAW_QUERIES.items():
    msgs = [{"role": "user", "content": q}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill
    PROMPTS[lang] = formatted
    tok_ids = tokenizer.encode(formatted, add_special_tokens=False)
    print(f"\n[{lang:<5}] Total Prompt Length: {len(tok_ids)} tokens")
    print(f"Tail: ...{formatted[-60:]!r}")

# Diagnostic verification of target tokens and candidate cities
CANDIDATE_CITIES = {
    "Cairo": " القاهرة",
    "Alexandria": " الإسكندرية",
    "Giza": " الجيزة",
    "Suez": " السويس",
    "Aswan": " أسوان",
    "Mansoura": " المنصورة",
    "Tanta": " طنطا",
}

print("\n=== Tokenizer Diagnostics: Candidate Egyptian Cities ===")
for name, word in CANDIDATE_CITIES.items():
    ids = tokenizer.encode(word, add_special_tokens=False)
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"{name:12} | Word: {repr(word):14} | IDs: {str(ids):25} | Decoded: {decoded}")


=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===

[EN   ] Total Prompt Length: 383 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>The capital is '

[MSA  ] Total Prompt Length: 381 tokens
Tail: ...'F_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>العاصمة هي '

[Masri] Total Prompt Length: 381 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>عاصمة البلد هي '

=== Tokenizer Diagnostics: Candidate Egyptian Cities ===
Cairo        | Word: ' القاهرة'     | IDs: [97936]                   | Decoded: [' القاهرة']
Alexandria   | Word: ' الإسكندرية'  | IDs: [4868, 45797, 2960, 14267] | Decoded: [' الإ', 'سك', 'ند', 'رية']
Giza         | Word: ' الجيزة'      | IDs: [153046]                  | Decoded: [' الجيزة']
Suez         | Word: ' السويس'      | IDs: [2658, 78092]             | Decoded: [' الس', 'ويس']
Aswan        | Word: ' أسوان'       | IDs: [11637, 6291]             | Decoded: [' أس', 'وان']
Mansoura     | Word: ' المنصورة'    | IDs: [9525, 7831

### Step 3: Pure PyTorch Weight-Only Quantization Engine (Int8, Int4 RTN, NF4)

To compare directly with Gemma-2-2B without platform-specific CUDA dependencies (like `bitsandbytes`), we implement a pure PyTorch weight-only quantization engine compatible with ROCm:
1. **Int8 (Symmetric Per-Tensor):** Standard 8-bit integer quantization mapping weights to $[-128, 127]$.
2. **Int4 RTN (Asymmetric Per-Group Round-To-Nearest):** 16 group bins with scale and zero-point ($block\_size = 128$).
3. **NF4 (NormalFloat4):** Quantile-optimal 4-bit representation for normally distributed neural weights ($block\_size = 64$), matching QLoRA's information-theoretic design.

A model wrapper dynamically swaps linear projection weights across precisions and restores unquantized FP16 weights without reloading from disk.

In [3]:
# Pure PyTorch Weight-Only Quantization Engine (No CUDA-only dependencies)

NF4_QUANT_TABLE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
], device=DEVICE, dtype=torch.float16)

def quantize_weight_int8(w: torch.Tensor) -> torch.Tensor:
    scale = w.abs().max() / 127.0
    w_q = torch.clamp(torch.round(w / scale), -128, 127).to(torch.int8)
    return (w_q.to(w.dtype) * scale).contiguous()

def quantize_weight_int4_rtn(w: torch.Tensor, block_size: int = 128) -> torch.Tensor:
    orig_shape = w.shape
    w_flat = w.reshape(-1, block_size)
    mn = w_flat.min(dim=-1, keepdim=True).values
    mx = w_flat.max(dim=-1, keepdim=True).values
    scale = torch.clamp((mx - mn) / 15.0, min=1e-8)
    q = torch.clamp(torch.round((w_flat - mn) / scale), 0, 15)
    return (q * scale + mn).reshape(orig_shape).contiguous()

def quantize_weight_nf4(w: torch.Tensor, block_size: int = 64) -> torch.Tensor:
    orig_shape = w.shape
    w_flat = w.reshape(-1, block_size)
    abs_max = torch.clamp(w_flat.abs().max(dim=-1, keepdim=True).values, min=1e-8)
    norm_w = w_flat / abs_max
    dists = torch.abs(norm_w.unsqueeze(-1) - NF4_QUANT_TABLE)
    best_idx = torch.argmin(dists, dim=-1)
    dequant = NF4_QUANT_TABLE[best_idx]
    return (dequant * abs_max).reshape(orig_shape).contiguous()

class ModelQuantizer:
    def __init__(self, model: torch.nn.Module):
        self.model = model
        self.original_weights = {}
        for name, module in self.model.named_modules():
            if isinstance(module, torch.nn.Linear) and "lm_head" not in name:
                self.original_weights[name] = module.weight.detach().clone()
        print(f"Quantizer initialized: cached {len(self.original_weights)} linear layers.")

    def apply_quantization(self, mode: str):
        if mode == "FP16":
            for name, orig_w in self.original_weights.items():
                dict(self.model.named_modules())[name].weight.data.copy_(orig_w)
            return

        for name, orig_w in self.original_weights.items():
            mod = dict(self.model.named_modules())[name]
            if mode == "Int8":
                mod.weight.data.copy_(quantize_weight_int8(orig_w))
            elif mode == "Int4 RTN":
                mod.weight.data.copy_(quantize_weight_int4_rtn(orig_w))
            elif mode == "NF4":
                mod.weight.data.copy_(quantize_weight_nf4(orig_w))
            else:
                raise ValueError(f"Unknown mode: {mode}")

    def restore_fp16(self):
        self.apply_quantization("FP16")


/tmp/ipykernel_259373/1944891803.py:3: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  NF4_QUANT_TABLE = torch.tensor([


### Step 4: Loading Tiny Aya Earth & Target Token Resolution

We load `CohereLabs/tiny-aya-earth` (3.35B, 36 layers) onto our ROCm accelerator in native `bfloat16`/`float16`. 

We register the target tokens corresponding to the single-token completions induced by our assistant prefill:
- **English Target:** `' Cairo'` (Token ID: `147633`)
- **MSA Target:** `' القاهرة'` (Token ID: `97936`)
- **Masri Target:** `' القاهرة'` (Token ID: `97936`)

We verify token resolution and initialize the `ModelQuantizer`.

In [4]:
MODEL_NAME = "CohereLabs/tiny-aya-earth"
print(f"Loading {MODEL_NAME} onto {DEVICE} in float16...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=DEVICE,
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Loaded successfully: {NUM_LAYERS} transformer layers.")
print(f"Logit scale: {getattr(model, 'logit_scale', 1.0)}")

TARGET_TOKENS = {
    "EN": " Cairo",
    "MSA": " القاهرة",
    "Masri": " القاهرة",
}

TARGET_IDS = {
    lang: tokenizer.encode(tok_str, add_special_tokens=False)[0]
    for lang, tok_str in TARGET_TOKENS.items()
}

print("\n=== Registered Target Tokens ===")
for lang, tid in TARGET_IDS.items():
    print(f"[{lang:<5}] Target Token ID: {tid:6d} | Decoded: {repr(tokenizer.decode([tid]))}")

quantizer = ModelQuantizer(model)


Loading CohereLabs/tiny-aya-earth onto cuda in float16...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded successfully: 36 transformer layers.
Logit scale: 1.0

=== Registered Target Tokens ===
[EN   ] Target Token ID: 147633 | Decoded: ' Cairo'
[MSA  ] Target Token ID:  97936 | Decoded: ' القاهرة'
[Masri] Target Token ID:  97936 | Decoded: ' القاهرة'
Quantizer initialized: cached 252 linear layers.


### Step 5: Multi-Layer Logit Lens Trajectory Across Precisions

We evaluate the prompt triples across **four precision regimes**:
1. **FP16 Baseline**
2. **Int8 Symmetric**
3. **Int4 RTN (Round-to-Nearest)**
4. **NF4 (NormalFloat4)**

For each precision and language, we extract the hidden states at all 36 layers at the final prefill sequence position.
We apply the final RMSNorm (`model.model.norm`) and project through the un-embedding matrix (`model.lm_head`) scaled by `model.logit_scale`. We compute the softmax probability assigned to the target token (`' Cairo'` or `' القاهرة'`) across layers.

In [5]:
records = []

def evaluate_precision(precision_label: str):
    print(f"\nEvaluating regime: {precision_label}...")
    quantizer.apply_quantization(precision_label)
    
    with torch.no_grad():
        for lang, prompt_text in PROMPTS.items():
            target_id = TARGET_IDS[lang]
            inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs, output_hidden_states=True)
            
            for layer_idx in range(NUM_LAYERS + 1):
                h = outputs.hidden_states[layer_idx][:, -1, :] # [1, hidden_dim]
                
                if layer_idx < NUM_LAYERS:
                    h_norm = model.model.norm(h)
                else:
                    h_norm = h
                    
                logits = model.lm_head(h_norm) * getattr(model, "logit_scale", 1.0)
                probs = torch.softmax(logits, dim=-1)
                target_p = probs[0, target_id].item()
                
                records.append({
                    "model": "tiny-aya-earth-3.35b",
                    "precision": precision_label,
                    "language": lang,
                    "layer": layer_idx,
                    "target_prob": target_p,
                    "target_token": TARGET_TOKENS[lang],
                    "target_id": target_id
                })

for prec in ["FP16", "Int8", "Int4 RTN", "NF4"]:
    evaluate_precision(prec)

quantizer.restore_fp16()

df_results = pd.DataFrame(records)
df_results.to_csv("quantization_tiny_aya_results.csv", index=False)
print("\nLogit lens evaluation complete. Saved results to quantization_tiny_aya_results.csv")

final_layer_summary = df_results[df_results["layer"] == NUM_LAYERS].pivot(
    index="precision", columns="language", values="target_prob"
)[["EN", "MSA", "Masri"]]
print("\n=== Final Layer Output Probabilities P(Cairo) ===")
print((final_layer_summary * 100).round(2).astype(str) + "%")



Evaluating regime: FP16...


/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/models/cohere2/modeling_cohere2.py:71: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash attention support on Navi31 GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at ../aten/src/ATen/native/transformers/hip/sdp_utils.cpp:225.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/yassermakram/code/fanous-llm-lens/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Memory Efficient attention on Navi31 GPU is still experimental. Enable it wi


Evaluating regime: Int8...



Evaluating regime: Int4 RTN...



Evaluating regime: NF4...



Logit lens evaluation complete. Saved results to quantization_tiny_aya_results.csv

=== Final Layer Output Probabilities P(Cairo) ===
language       EN     MSA   Masri
precision                        
FP16       50.05%  32.28%  21.85%
Int4 RTN   26.95%  13.77%  11.46%
Int8        64.4%  31.13%  21.25%
NF4        42.36%  22.84%  12.85%


### Step 5b: Factual Grounding & Candidate City Contrastive Discrimination

To verify beyond doubt that the model is performing genuine factual retrieval rather than relying on unigram frequency or definite-article bias, we evaluate the full probability distribution over competing Egyptian cities at the final layer across all four quantization regimes:
- **Target:** `' القاهرة'` (Cairo)
- **Competitors:** `' الإسكندرية'` (Alexandria), `' الجيزة'` (Giza), `' السويس'` (Suez), `' أسوان'` (Aswan), `' المنصورة'` (Mansoura), `' طنطا'` (Tanta)

We compute both the direct next-token probability and the **Contrast Ratio** $\frac{P(\text{Cairo})}{P(\text{Alexandria})}$. A ratio $\gg 1$ across all precisions confirms robust factual resolution.

In [6]:
candidate_records = []

with torch.no_grad():
    for prec in ["FP16", "Int8", "Int4 RTN", "NF4"]:
        quantizer.apply_quantization(prec)
        for lang in ["MSA", "Masri"]:
            prompt_text = PROMPTS[lang]
            inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :] * getattr(model, "logit_scale", 1.0)
            probs = torch.softmax(logits, dim=-1)
            
            p_cairo = probs[TARGET_IDS[lang]].item()
            
            for city_name, city_word in CANDIDATE_CITIES.items():
                cand_id = tokenizer.encode(city_word, add_special_tokens=False)[0]
                p_cand = probs[cand_id].item()
                candidate_records.append({
                    "precision": prec,
                    "language": lang,
                    "city": city_name,
                    "first_token": tokenizer.decode([cand_id]),
                    "prob": p_cand,
                    "contrast_vs_cairo": (p_cairo / p_cand) if p_cand > 0 else float("inf")
                })

quantizer.restore_fp16()

df_candidates = pd.DataFrame(candidate_records)
df_candidates.to_csv("candidate_city_discrimination.csv", index=False)

print("=== Candidate City Contrastive Analysis (FP16 Baseline) ===")
for lang in ["MSA", "Masri"]:
    sub = df_candidates[(df_candidates["precision"] == "FP16") & (df_candidates["language"] == lang)]
    sub_sorted = sub.sort_values(by="prob", ascending=False)
    print(f"\n--- {lang} City Ranking at Output Frontier ---")
    for _, row in sub_sorted.iterrows():
        print(f"{row['city']:12} | Token: {repr(row['first_token']):12} | P: {row['prob']*100:6.2f}% | Cairo Contrast: {row['contrast_vs_cairo']:6.1f}x")


=== Candidate City Contrastive Analysis (FP16 Baseline) ===

--- MSA City Ranking at Output Frontier ---
Cairo        | Token: ' القاهرة'   | P:  32.28% | Cairo Contrast:    1.0x
Alexandria   | Token: ' الإ'       | P:   1.08% | Cairo Contrast:   29.9x
Suez         | Token: ' الس'       | P:   0.06% | Cairo Contrast:  578.0x
Aswan        | Token: ' أس'        | P:   0.03% | Cairo Contrast: 1083.8x
Tanta        | Token: ' طن'        | P:   0.02% | Cairo Contrast: 1571.4x
Mansoura     | Token: ' المن'      | P:   0.01% | Cairo Contrast: 4936.1x
Giza         | Token: ' الجيزة'    | P:   0.00% | Cairo Contrast: 21659.6x

--- Masri City Ranking at Output Frontier ---
Cairo        | Token: ' القاهرة'   | P:  21.85% | Cairo Contrast:    1.0x
Alexandria   | Token: ' الإ'       | P:   2.15% | Cairo Contrast:   10.2x
Suez         | Token: ' الس'       | P:   0.11% | Cairo Contrast:  204.4x
Tanta        | Token: ' طن'        | P:   0.08% | Cairo Contrast:  287.3x
Aswan        | Token: ' أس'      

### Step 6: Quantitative Degradation Analysis & Head-to-Head Comparison with Gemma-2-2B

We compute the relative percentage change $\Delta\%$ in final target probability relative to the unquantized FP16 baseline:
$$\Delta\% = \frac{P_{\text{quant}} - P_{\text{FP16}}}{P_{\text{FP16}}} \times 100$$

We load the `quantization_multihop_results.csv` baseline from Gemma-2-2B and compare how both architectures handle low-bit quantization in English, Modern Standard Arabic, and Egyptian Arabic.

In [2]:
# Load Gemma-2-2B baseline results for direct head-to-head comparison
df_gemma_raw = pd.read_csv("quantization_multihop_results.csv")
df_aya = pd.read_csv("quantization_tiny_aya_results.csv")

# Filter Gemma-2 to final layer (layer_num == 26)
gemma_last_layer = df_gemma_raw["layer_num"].max()
gemma_final = df_gemma_raw[df_gemma_raw["layer_num"] == gemma_last_layer].copy()

# Map precision names to standard: 'FP16', 'Int8', 'Int4 RTN'
prec_map = {
    "FP16 Baseline": "FP16",
    "Int8 Quantized": "Int8",
    "Int4 Quantized": "Int4 RTN"
}
gemma_final["precision"] = gemma_final["precision"].map(prec_map)
gemma_final["language"] = gemma_final["lang"]
gemma_final["target_prob"] = gemma_final.apply(
    lambda r: r["p_cairo_en"] if r["lang"] == "EN" else r["p_cairo_ar"], axis=1
)
gemma_final["model"] = "Gemma-2-2B (2.6B)"

# Tiny Aya final layer (layer 36)
NUM_LAYERS = 36
aya_final = df_aya[df_aya["layer"] == NUM_LAYERS].copy()
aya_final["model"] = "Tiny Aya Earth (3.35B)"

def compute_deltas(df_final):
    fp16_map = df_final[df_final["precision"] == "FP16"].set_index("language")["target_prob"].to_dict()
    df_final["fp16_baseline"] = df_final["language"].map(fp16_map)
    df_final["delta_pct"] = ((df_final["target_prob"] - df_final["fp16_baseline"]) / df_final["fp16_baseline"]) * 100.0
    return df_final

gemma_summary = compute_deltas(gemma_final)
aya_summary = compute_deltas(aya_final)

comparison_records = []
for m_label, df_m in [("Gemma-2-2B (2.6B)", gemma_summary), ("Tiny Aya Earth (3.35B)", aya_summary)]:
    for _, row in df_m.iterrows():
        comparison_records.append({
            "Model": m_label,
            "Precision": row["precision"],
            "Language": row["language"],
            "P(Target)": round(row["target_prob"], 4),
            "Delta vs FP16 (%)": round(row["delta_pct"], 1)
        })

df_comp = pd.DataFrame(comparison_records)
df_comp.to_csv("gemma2_vs_tiny_aya_quantization_comparison.csv", index=False)

print("=== Head-to-Head Quantization Resilience Comparison ===")
pivot_p = df_comp.pivot(index=["Model", "Precision"], columns="Language", values="P(Target)")
pivot_delta = df_comp.pivot(index=["Model", "Precision"], columns="Language", values="Delta vs FP16 (%)")

display_df = pivot_p.astype(str) + " (" + pivot_delta.map(lambda x: f"{x:+.1f}%") + ")"
print(display_df[["EN", "MSA", "Masri"]])


=== Head-to-Head Quantization Resilience Comparison ===
Language                                       EN              MSA  \
Model                  Precision                                     
Gemma-2-2B (2.6B)      FP16         0.964 (+0.0%)   0.3786 (+0.0%)   
                       Int4 RTN   0.6904 (-28.4%)  0.1296 (-65.8%)   
                       Int8        0.9615 (-0.3%)  0.3088 (-18.5%)   
Tiny Aya Earth (3.35B) FP16        0.5005 (+0.0%)   0.3228 (+0.0%)   
                       Int4 RTN   0.2695 (-46.1%)  0.1377 (-57.3%)   
                       Int8        0.644 (+28.7%)   0.3113 (-3.6%)   
                       NF4        0.4236 (-15.4%)  0.2284 (-29.2%)   

Language                                    Masri  
Model                  Precision                   
Gemma-2-2B (2.6B)      FP16        0.2382 (+0.0%)  
                       Int4 RTN   0.0646 (-72.9%)  
                       Int8        0.2215 (-7.0%)  
Tiny Aya Earth (3.35B) FP16        0.2185 (+0.0%)  
 